In [3]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1. Setup Paths (Going up one level from 'notebooks' to root)
root = Path().resolve().parent
load_dotenv(root / ".env")

# 2. Get Credentials
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")
ONET_DIR = root / "data" / "raw" / "onet"

# 3. Initialize the Driver
driver = GraphDatabase.driver(URI, auth=(USER, PWD))

def run_query(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters)
        return [record.data() for record in result]

print("✅ Connection Bridge Ready!")

✅ Connection Bridge Ready!


In [4]:
try:
    df_occ = pd.read_csv(ONET_DIR / "Occupation Data.txt", sep='\t')
    df_skills = pd.read_csv(ONET_DIR / "Skills.txt", sep='\t')
    df_tasks = pd.read_csv(ONET_DIR / "Task Statements.txt", sep='\t')
    
    print(f"📊 Data Loaded Successfully:")
    print(f"- {len(df_occ)} Occupations")
    print(f"- {len(df_skills)} Skill data rows")
    print(f"- {len(df_tasks)} Task statements")
except Exception as e:
    print(f"❌ Error loading files: {e}")

📊 Data Loaded Successfully:
- 1016 Occupations
- 62580 Skill data rows
- 18796 Task statements


In [5]:
query_occ = """
UNWIND $rows AS row
MERGE (o:Occupation {code: row.code})
SET o.title = row.title,
    o.description = row.description
"""

rows_occ = [
    {"code": r['O*NET-SOC Code'], "title": r['Title'], "description": r['Description']} 
    for _, r in df_occ.iterrows()
]

run_query(query_occ, {"rows": rows_occ})
print("✅ Occupation nodes are live in AuraDB!")

✅ Occupation nodes are live in AuraDB!


In [6]:
# 1. Create unique Skill nodes
query_skill_nodes = """
UNWIND $rows AS row
MERGE (s:Skill {id: row.id})
SET s.name = row.name
"""
unique_skills = df_skills[['Element ID', 'Element Name']].drop_duplicates()
rows_s = [{"id": r['Element ID'], "name": r['Element Name']} for _, r in unique_skills.iterrows()]
run_query(query_skill_nodes, {"rows": rows_s})

# 2. Pivot data and create weighted relationships
skills_pivot = df_skills.pivot_table(
    index=['O*NET-SOC Code', 'Element ID'], 
    columns='Scale ID', 
    values='Data Value'
).reset_index()

query_skill_rel = """
UNWIND $rows AS row
MATCH (o:Occupation {code: row.occ})
MATCH (s:Skill {id: row.skill})
MERGE (o)-[r:REQUIRES]->(s)
SET r.importance = toFloat(row.IM),
    r.level = toFloat(row.LV)
"""
rows_rel = [
    {"occ": r['O*NET-SOC Code'], "skill": r['Element ID'], "IM": r.get('IM', 0), "LV": r.get('LV', 0)} 
    for _, r in skills_pivot.iterrows()
]

run_query(query_skill_rel, {"rows": rows_rel})
print("✅ Skills mapped to Occupations with weights!")

✅ Skills mapped to Occupations with weights!


In [7]:
query_task = """
UNWIND $rows AS row
MATCH (o:Occupation {code: row.occ_code})
MERGE (t:Task {id: row.id})
SET t.statement = row.task
MERGE (o)-[:PERFORMS]->(t)
"""

rows_task = [
    {"occ_code": r['O*NET-SOC Code'], "id": r['Task ID'], "task": r['Task']} 
    for _, r in df_tasks.iterrows()
]

run_query(query_task, {"rows": rows_task})
print("✅ Tasks ingested and linked!")

✅ Tasks ingested and linked!


In [8]:
check_query = """
MATCH (n)
RETURN labels(n) as Label, count(n) as Count
"""
stats = run_query(check_query)
for stat in stats:
    print(f"Node Type: {stat['Label'][0]} | Total: {stat['Count']}")

Node Type: Occupation | Total: 1016
Node Type: Skill | Total: 35
Node Type: Task | Total: 18796
